In [3]:
import struct
import numpy as np
import math
from numpy.random import *
import WinoTran_NCHW as NCHW

In [3]:
#generation
# chn = 512
# numOfFilter =512
# # print(224*224*64)
# # print(112*112*128)
# # print(56*56*256)
# # print(28*28*512)
# parameter = chn *3*3* numOfFilter

# input1 = (np.array(rand(parameter))-0.5).astype(np.float32)
# des = open("kernel.bin","wb")
# cnt = des.write(input1)
# des.close()

In [70]:
bat4Conv =1
inside = 224
chn = 1
numOfFilter =64
padding =1
inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)
M = (int)(blockn * blockn * bat4Conv);
N = numOfFilter;
K = chn

MSize = M if (M%128 == 0) else math.ceil(M/128)*128
NSize = N if (N%128 == 0) else math.ceil(N/128)*128
KSize = (int((K-1)/8)+1)*8

parameters1 = bat4Conv * inside * inside * chn
parameters2 = numOfFilter * 3 * 3 * chn

# readin the feature map
src = open("../../M1/data/input.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_input = input.reshape((bat4Conv,chn,inside,inside)).astype(np.float32)

inputTran1 = NCHW.Wino_inputTran(sample_input,padding)

# readin the filter data
src = open("../../M2/data/kernel.bin","rb")
context = src.read(parameters2*4)
real_context = struct.unpack(str(parameters2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_kernel = input.reshape((numOfFilter,chn,3,3)).astype(np.float32)
print(sample_kernel.shape)

kernelTran1 = NCHW.Wino_kernelTran(sample_kernel)


(64, 1, 3, 3)


In [67]:
pyout = NCHW.gemm(inputTran1, kernelTran1, MSize, NSize, False)

In [68]:
for i in range(0,1):
    parameter2 = 36* MSize*NSize
    print(MSize,NSize)
    src = open("./gemmOut.bin","rb")
    context = src.read(parameter2*4)
    real_context = struct.unpack(str(parameter2)+'f',context)
    input = np.array(real_context)
    # input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
    testoutput = input.reshape((36,MSize,NSize)).astype(np.float32)
    print( np.sum(np.abs(pyout[:,:,:]-testoutput[:,:,:])) )

3200 128
0.10193595


In [69]:
# readin the feature map
parameters1 = 36*MSize*KSize
src = open("../../M1/data/M1_orig.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
inputTran = input.reshape((36,KSize, MSize)).astype(np.float32)

# inputTran = NCHW.Wino_inputTran(sample_input,padding)


parameters2 = 36*NSize*KSize
# readin the filter data
src = open("../../M2/data/M2_new0.bin","rb")
context = src.read(parameters2*4)
real_context = struct.unpack(str(parameters2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
kernelTran = input.reshape((36,KSize, NSize)).astype(np.float32)
# print(sample_kernel.shape)

# kernelTran = NCHW.Wino_kernelTran(sample_kernel)

pyout = NCHW.gemm(inputTran, kernelTran, MSize, NSize, False)